# A coastal boundary for the synthetic valley

The synthetic valley drains south. Its southern edge is closed except where the
river leaves the domain, so every drop of recharge that is not pumped or lost to
evapotranspiration has to reach the river. Put the sea along that edge instead
and the valley becomes a coastal aquifer, with the river and the coast competing
for the same water.

## What this notebook covers

Add a **general-head boundary** (**GHB**) at sea level along the southern row of
the valley, run the model with and without it, and measure what the coast takes
from the river. Then repeat the comparison on the advanced valley, where the
river is a streamflow-routing (**SFR**) network and the lake, unsaturated zone,
and water mover are all active.

By the end of this notebook you will be able to:

- build a GHB along a model edge from the grid and conductivity arrays,
- say what the `ELEVATION` and `CONCENTRATION` auxiliary variables are for,
- compare two simulations through their budget files and observation output, and
- read the discharge split between a stream and a coast from those budgets.

A GHB is a head-dependent boundary: it moves water into or out of a cell in
proportion to the difference between a fixed boundary head and the simulated
head in the cell, scaled by a **conductance** that carries the units of the
sediment between them. The sea is the natural thing to represent this way,
because its level is set by something far larger than the aquifer.

Import the packages this notebook uses. `mf6_notebook_helpers` holds the shared
synthetic-valley setup, including the coastal boundary builder used below.

In [ ]:
import pathlib as pl
import shutil

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mf6_notebook_helpers import coastal_ghb_data, find_mf6_libraries

Locate the MODFLOW 6 executable in the active environment.

In [ ]:
_, mf6_exe = find_mf6_libraries()
print(f"executable: {mf6_exe.name}")

## Look at the valley before the sea

Load the calibrated base model. It is a 5-layer, 40-row, 25-column grid of
500-foot cells in feet and days, with recharge (**RCH**), evapotranspiration
(**EVT**), a river (**RIV**) running down column 9, two production wells, and a
prediction well that starts pumping in stress period 12. The lake in the north
is represented by cells of very high hydraulic conductivity rather than by a
lake package.

In [ ]:
data_root = pl.Path("../data/synthetic-valley")
base_ws = data_root / "synthetic-valley-base-annual"

sim = flopy.mf6.MFSimulation.load(sim_ws=str(base_ws), verbosity_level=0)
gwf = sim.get_model()
nlay, nrow, ncol = gwf.dis.nlay.data, gwf.dis.nrow.data, gwf.dis.ncol.data
south = nrow - 1  # zero-based index of the southern row
outlet_col = 8  # zero-based column the river leaves the domain in

top_south = gwf.dis.top.array[south]
print(f"grid:            {nlay} layers, {nrow} rows, {ncol} columns")
print(f"stress periods:  {sim.tdis.nper.data}")
print(
    f"land surface along the southern row: {top_south.min():.1f} to {top_south.max():.1f} ft"
)

Map the valley to see where the sea will go. Plot the river cells with
`.plot_bc()`, mark the southern row, and mark the cell the river leaves through.
The figure is drawn inside `flopy.plot.styles.USGSMap()`, which supplies the
fonts and tick geometry used throughout the training material.

In [ ]:
xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(4.5, 7), layout="constrained")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    mm.plot_grid(lw=0.2, color="0.85")
    mm.plot_bc("RIV", color="tab:cyan")
    coast_cols = [j for j in range(ncol) if j != outlet_col]
    ax.plot(
        xc[south, coast_cols],
        yc[south, coast_cols],
        "s",
        color="tab:blue",
        ms=6,
        label="coast",
    )
    ax.plot(
        xc[south, outlet_col],
        yc[south, outlet_col],
        "s",
        color="tab:red",
        ms=6,
        label="river outlet",
    )
    ax.plot([], [], "s", color="tab:cyan", label="river")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_xlabel("x, in feet")
    ax.set_ylabel("y, in feet")
    ax.set_title("Southern edge of the synthetic valley")

**What to look for.** The river runs the length of the valley down column 9 and
leaves through the red cell in the southern row. Every other cell in that row
(blue) is closed today and becomes part of the coast. Land surface along the row
runs 2.2 to 8.3 ft, so a sea at 0 ft sits below the valley floor.

## Build the coastal boundary

A GHB is a list package, so each entry is a tuple. The valley models use
boundary names and two auxiliary variables, which puts the fields in this order:

```python
# ((layer, row, column), bhead, cond, elevation, concentration, boundname)
((4, 39, 0), 0.0, 52477.1, -235.7, 35.0, "coast")
```

`bhead` is sea level, the same 0.0 ft in every layer. MODFLOW 6 solves in
hydraulic head, and a column of seawater whose surface sits at sea level has
that same head at every depth, so no depth correction belongs in the input.

`ELEVATION` is the middle of the submerged part of the cell face and
`CONCENTRATION` is the salinity of seawater, 35 kg/m3. Both are inert in this
notebook: they matter only once the Buoyancy (**BUY**) package is active and
density varies, which is a later notebook. Carrying them from the start means
the same boundary serves both models.

Conductance is the one value that has to be computed per cell. Take it as the
conductance of the cell itself across its outer face - hydraulic conductivity
times the face area, divided by the distance from the cell center to that face.
`coastal_ghb_data()` does this for every active cell in the row, skipping the
outlet column so the river keeps the only surface outflow there.

In [ ]:
# exercise: build the GHB data for the southern row with coastal_ghb_data()
# your code here

print(f"boundary cells:  {len(ghb_spd)} ({nlay} layers x {ncol - 1} columns)")
print(f"auxiliary:       {', '.join(ghb_aux)}")
print(
    f"conductance:     {min(c[2] for c in ghb_spd):,.0f} to {max(c[2] for c in ghb_spd):,.0f} ft2/d"
)
print(f"total:           {sum(c[2] for c in ghb_spd):,.0f} ft2/d")
print(f"deepest cell:    {ghb_spd[-1]}")

**What to look for.** The 120 boundary cells carry 2.2 million ft2/d of
conductance between them, and layers 4 and 5 hold most of it because they are
the thickest. For comparison, the river's conductance totals 1.8 million ft2/d,
so the coast is the better-connected of the two sinks.

## Run the valley with and without the sea

Copy the shipped model twice. One copy is left alone, and the other gets the GHB
package built above. Everything else - recharge, evapotranspiration, the river,
the wells - is identical, so any difference between the two runs belongs to the
coast.

In [ ]:
model_root = pl.Path("models")
no_coast_ws = model_root / "coastal-ghb-no-coast"
coastal_ws = model_root / "coastal-ghb-base"

for ws in (no_coast_ws, coastal_ws):
    if ws.exists():
        shutil.rmtree(ws)
    shutil.copytree(base_ws, ws)

sim = flopy.mf6.MFSimulation.load(sim_ws=str(coastal_ws), verbosity_level=0)
gwf_coastal = sim.get_model()
flopy.mf6.ModflowGwfghb(
    gwf_coastal,
    pname="ghb-1",
    auxiliary=ghb_aux,
    boundnames=True,
    stress_period_data={0: ghb_spd},
)
sim.write_simulation(silent=True)

for ws in (no_coast_ws, coastal_ws):
    success, buff = flopy.run_model(
        exe_name=str(mf6_exe), namefile=None, model_ws=str(ws), silent=True
    )
    if not success:
        raise RuntimeError("\n".join(buff[-15:]))
print("both simulations converged")

Read the last stress period of each budget file. `sv-budget.csv` carries one
column per package for flow in and flow out, so the net discharge to a boundary
is the difference between them.

In [ ]:
def net_flow(ws, term):
    """Net flow out of the aquifer to one package in the last stress period."""
    budget = pd.read_csv(pl.Path(ws) / "sv-budget.csv").iloc[-1]
    return budget[f"{term}_OUT"] - budget[f"{term}_IN"]


river_before = net_flow(no_coast_ws, "RIV(RIV-1)")
river_after = net_flow(coastal_ws, "RIV(RIV-1)")
coast_after = net_flow(coastal_ws, "GHB(GHB-1)")

print(f"river, no coast: {river_before:12,.0f} ft3/d")
print(
    f"river, coastal:  {river_after:12,.0f} ft3/d ({100 * river_after / river_before:.0f} percent retained)"
)
print(f"coast, coastal:  {coast_after:12,.0f} ft3/d")
print(
    f"coast share:     {100 * coast_after / (coast_after + river_after):12.0f} percent"
)

Plot what the sea does to the water table. Panel A follows the head along the
southern row in both runs, and panel B maps how much the head falls when the
coast is added.

In [ ]:
head_before = flopy.utils.HeadFile(no_coast_ws / "sv.hds").get_alldata()[-1]
head_after = flopy.utils.HeadFile(coastal_ws / "sv.hds").get_alldata()[-1]
drawdown = head_before[0] - head_after[0]

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic([["A", "B"]], figsize=(9.5, 6), layout="constrained")

    ax = axd["A"]
    ax.plot(
        np.arange(1, ncol + 1),
        head_before[0, south],
        "o-",
        color="tab:gray",
        label="no coast",
    )
    ax.plot(
        np.arange(1, ncol + 1),
        head_after[0, south],
        "o-",
        color="tab:blue",
        label="coastal",
    )
    ax.axhline(0.0, lw=0.8, ls=":", color="k", label="sea level")
    ax.set_xlabel("column")
    ax.set_ylabel("head, in feet")
    ax.legend(fontsize=8)
    ax.set_title("A. Head along the southern row")

    ax = axd["B"]
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    cb = mm.plot_array(drawdown, cmap="magma_r")
    mm.plot_bc("RIV", color="tab:cyan")
    fig.colorbar(cb, ax=ax, shrink=0.5, label="head decline, in feet")
    ax.set_xlabel("x, in feet")
    ax.set_ylabel("y, in feet")
    ax.set_title("B. Head decline in layer 1")

**What to look for.** In panel A the head along the southern row falls from
0.24-2.79 ft to 0.01-0.16 ft, which is the coast holding that edge at sea level.
Panel B shows the decline reaching back through the valley: it is 2.64 ft at
most, near the southern edge, and averages 0.49 ft over the grid, so the lake
end of the valley barely moves.

The budget above is the result to keep: the coast takes 134,005 ft3/d and the
river keeps 269,121 ft3/d of the 403,622 ft3/d it carried before, which is two
thirds of it. The coast has more conductance than the river but gets only a
third of the water, because the river sits in the middle of the valley where the
water table is high and the coast only drains the southern end.

## The advanced valley

The advanced valley replaces the river with an SFR network, the high-conductivity
lake cells with a lake (**LAK**) package, recharge and evapotranspiration with
unsaturated-zone flow (**UZF**), and the production wells with multi-aquifer
wells (**MAW**). A water mover (**MVR**) routes rejected infiltration from UZF to
the lake and the stream. The same coastal boundary is already built into the
shipped coastal dataset, so load it and its no-coast twin and run both.

In [ ]:
adv_no_coast_ws = model_root / "coastal-ghb-adv-no-coast"
adv_coastal_ws = model_root / "coastal-ghb-adv"

for ws, src in (
    (adv_no_coast_ws, data_root / "synthetic-valley-advanced-annual"),
    (adv_coastal_ws, data_root / "synthetic-valley-coastal-advanced-annual"),
):
    if ws.exists():
        shutil.rmtree(ws)
    shutil.copytree(src, ws)
    success, buff = flopy.run_model(
        exe_name=str(mf6_exe), namefile=None, model_ws=str(ws), silent=True
    )
    if not success:
        raise RuntimeError("\n".join(buff[-15:]))
print("both simulations converged")

The stream and the lake both write observation files, so read the gauge flow and
the lake stage from those rather than from the budget.

In [ ]:
def stream_gauge(ws):
    """Streamflow at the downstream gauge in the last stress period, in ft3/d."""
    return -pd.read_csv(pl.Path(ws) / "sv.sfr.obs.csv").iloc[-1]["RIV-FLOW"]


def lake_stage(ws):
    """Lake stage at the end of the simulation, in feet."""
    return pd.read_csv(pl.Path(ws) / "sv.lake.obs.csv").iloc[-1]["LAKE-STAGE"]


print(f"gauge flow, no coast: {stream_gauge(adv_no_coast_ws):12,.0f} ft3/d")
print(f"gauge flow, coastal:  {stream_gauge(adv_coastal_ws):12,.0f} ft3/d")
print(f"coast discharge:      {net_flow(adv_coastal_ws, 'GHB(GHB-1)'):12,.0f} ft3/d")
print(f"lake stage, no coast: {lake_stage(adv_no_coast_ws):12.4f} ft")
print(f"lake stage, coastal:  {lake_stage(adv_coastal_ws):12.4f} ft")

**What to look for.** The stream keeps more of its water than the river did:
692,139 ft3/d at the gauge against 899,588 ft3/d without the coast, or 77
percent, while the coast takes 276,026 ft3/d. The stream is fed along its whole
length by groundwater discharge and by water the mover routes to it, so it
competes with the coast better than a river reach does.

The lake stage falls 0.09 ft, from 13.3467 to 13.2533 ft, even though the lake
is 27 rows from the coast. The coast lowers the water table across the valley,
which leaves less rejected infiltration for the mover to route into the lake, so
the lake receives less water and its surface drops. The lake is coupled to the
coast through the unsaturated zone rather than through the aquifer.

## Recap

- A GHB at sea level along the southern row turns the valley into a coastal
  aquifer. It holds that edge within 0.2 ft of sea level and lowers heads
  through the valley by 0.49 ft on average.
- Conductance comes from the grid and the conductivity arrays, one value per
  cell, and the outlet column is left out so the river or stream keeps the only
  surface outflow there.
- The coast takes a third of the base model's discharge and about a quarter of
  the advanced model's, so the stream survives the change with most of its flow.
- The `ELEVATION` and `CONCENTRATION` auxiliary variables do nothing here. They
  are what the Buoyancy package reads once density varies, which is where the
  coastal valley goes next.